# Forecast Error Analysis — UK Wind Power

This notebook analyzes the **error characteristics** of the UK wind generation forecast (WINDFOR) against actual half-hourly generation (FUELHH, WIND).

**Data**: Elexon BMRS — actuals: FUELHH (fuelType=WIND); forecasts: WINDFOR. From January 2025 onwards. Forecast horizon considered: 0–48 hours.

**Goals**:
- Mean, median, and p99 absolute error (and signed error)
- How error varies with forecast horizon
- How error varies by time of day

## 1. Assumptions and data loading

- **Matching rule**: For each target time (actual), we pair it with the *latest* forecast that has the same `startTime` and was *published* at least 4 hours before the target (configurable). We only use forecasts whose horizon (startTime − publishTime) is between 0 and 48 hours.
- **Missing data**: Points with no matching forecast are dropped for error metrics; we do not impute.
- **Units**: All generation values in MW.

We load actuals and forecasts from the Elexon stream API (or from pre-downloaded CSVs if the API is unavailable).

ModuleNotFoundError: No module named 'pandas'

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timezone
import requests

BMRS_BASE = "https://data.elexon.co.uk/bmrs/api/v1"
JAN_2025 = "2025-01-01T00:00:00Z"

def fetch_fuelhh(from_ts: str, to_ts: str):
    url = f"{BMRS_BASE}/datasets/FUELHH/stream?from={from_ts}&to={to_ts}"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    data = r.json()
    if isinstance(data, dict) and "data" in data:
        data = data["data"]
    if not isinstance(data, list):
        return pd.DataFrame()
    df = pd.DataFrame(data)
    df = df[df["fuelType"] == "WIND"].copy()
    df["startTime"] = pd.to_datetime(df["startTime"], utc=True)
    df["generation"] = pd.to_numeric(df["generation"], errors="coerce").fillna(0)
    return df[["startTime", "generation"]].rename(columns={"generation": "actual"})

def fetch_windfor(from_ts: str, to_ts: str):
    url = f"{BMRS_BASE}/datasets/WINDFOR/stream?from={from_ts}&to={to_ts}"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    data = r.json()
    if isinstance(data, dict) and "data" in data:
        data = data["data"]
    if not isinstance(data, list):
        return pd.DataFrame()
    df = pd.DataFrame(data)
    df["startTime"] = pd.to_datetime(df["startTime"], utc=True)
    df["publishTime"] = pd.to_datetime(df["publishTime"], utc=True)
    df["generation"] = pd.to_numeric(df["generation"], errors="coerce").fillna(0)
    return df[["startTime", "publishTime", "generation"]].rename(columns={"generation": "forecast"})

# Example: last 14 days of data (adjust dates if API returns different range)
from_dt = "2025-01-15T00:00:00Z"
to_dt   = "2025-01-29T23:59:59Z"

try:
    actuals = fetch_fuelhh(from_dt, to_dt)
    forecasts = fetch_windfor(from_dt, to_dt)
    print("Actuals:", len(actuals), "rows")
    print("Forecasts:", len(forecasts), "rows")
except Exception as e:
    print("API fetch failed:", e)
    # Fallback: use synthetic data for structure demo
    actuals = pd.DataFrame({
        "startTime": pd.date_range(from_dt, periods=672, freq="30min", tz="UTC"),
        "actual": np.clip(np.random.randn(672).cumsum() * 200 + 4000, 0, 15000)
    })
    forecasts = pd.DataFrame({
        "startTime": pd.date_range(from_dt, periods=672, freq="30min", tz="UTC"),
        "publishTime": pd.date_range(from_dt, periods=672, freq="30min", tz="UTC") - pd.Timedelta(hours=4),
        "forecast": actuals["actual"].values + np.random.randn(672) * 200
    })
    print("Using synthetic data for demo.")
    print("Actuals:", len(actuals), "rows")
    print("Forecasts:", len(forecasts), "rows")

## 2. Build matched pairs (latest forecast per target, horizon ≥ 4h, 0–48h)

For each target time we keep the forecast with that `startTime` and the **latest** `publishTime` among those with horizon in [4, 48] hours (min horizon 4h for “at least 4 hours before”).

In [ ]:
HORIZON_MIN_H = 4
HORIZON_MAX_H = 48

forecasts["horizon_h"] = (forecasts["startTime"] - forecasts["publishTime"]).dt.total_seconds() / 3600
valid = forecasts[(forecasts["horizon_h"] >= HORIZON_MIN_H) & (forecasts["horizon_h"] <= HORIZON_MAX_H)]

# Latest forecast per startTime (max publishTime)
latest = valid.sort_values("publishTime").groupby("startTime", as_index=False).last()
latest = latest[["startTime", "forecast", "horizon_h"]].rename(columns={"horizon_h": "horizon_hours"})

merged = actuals.merge(latest, on="startTime", how="inner")
merged["error"] = merged["actual"] - merged["forecast"]
merged["abs_error"] = merged["error"].abs()
merged["hour"] = merged["startTime"].dt.hour
merged["time_of_day"] = merged["startTime"].dt.strftime("%H:%M")

print("Matched pairs:", len(merged))
merged.head(10)

## 3. Mean, median, and p99 error

- **Signed error** = actual − forecast (positive ⇒ under-forecast).
- **Absolute error** = |actual − forecast|.

In [ ]:
print("=== Signed error (actual - forecast), MW ===")
print("Mean:", round(float(merged["error"].mean()), 2))
print("Median:", round(float(merged["error"].median()), 2))
print("Std:", round(float(merged["error"].std()), 2))

print("\n=== Absolute error, MW ===")
print("Mean (MAE):", round(float(merged["abs_error"].mean()), 2))
print("Median:", round(float(merged["abs_error"].median()), 2))
print("P99:", round(float(merged["abs_error"].quantile(0.99)), 2))
print("P95:", round(float(merged["abs_error"].quantile(0.95)), 2))

print("\n=== Summary ===")
merged["abs_error"].describe()

## 4. Error vs forecast horizon

We bin by horizon (e.g. 4–8h, 8–12h, …) and compute MAE and median absolute error per bin to see if error grows with lead time.

In [ ]:
merged["horizon_bin"] = (merged["horizon_hours"] // 4).astype(int) * 4  # 4, 8, 12, ...
by_horizon = merged.groupby("horizon_bin").agg(
    mae=("abs_error", "mean"),
    median_ae=("abs_error", "median"),
    p99_ae=("abs_error", lambda s: s.quantile(0.99)),
    count=("abs_error", "count")
).round(2)
by_horizon

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(by_horizon.index.astype(str) + "h", by_horizon["mae"], label="MAE", color="steelblue", alpha=0.8)
ax.set_xlabel("Forecast horizon (hours)")
ax.set_ylabel("MAE (MW)")
ax.set_title("Mean absolute error by forecast horizon")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Error by time of day

Average absolute error by hour of day to see if certain periods (e.g. morning ramp, evening) are harder to forecast.

In [ ]:
by_hour = merged.groupby("hour").agg(
    mae=("abs_error", "mean"),
    median_ae=("abs_error", "median"),
    count=("abs_error", "count")
).round(2)
by_hour

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(by_hour.index, by_hour["mae"], marker="o", label="MAE by hour")
ax.set_xlabel("Hour of day (UTC)")
ax.set_ylabel("MAE (MW)")
ax.set_title("Mean absolute error by time of day")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Conclusions

- **Central tendency**: Mean/median signed error indicate whether the forecast is biased (e.g. systematically under- or over-forecasting).
- **Spread**: MAE, median AE, and P99 describe typical and tail errors; P99 is useful for reserve/planning.
- **Horizon**: If MAE increases with horizon, shorter-lead forecasts are more reliable.
- **Time of day**: Peaks in MAE by hour may align with ramps or weather transitions; this can inform when to add extra reserves or use other flexibility.